## RAG: Adam & Isaac

Below, we take a [sample employee handbook](https://www.501commons.org/resources/tools-and-best-practices/human-resources/sample-employee-handbook-national-council-of-nonprofits) from the web and prepare it for use in a RAG application. We're using a Small-to-Big approach, where each embedding represents one paragraph, but is associated with (and will return) that paragraph and the ones preceding and following it.

In [3]:
!pip install duckdb FlagEmbedding pymupdf4llm httpx

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.9/163.9 kB 14.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.1/24.1 MB 98.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 866.1/866.1 kB 47.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 149.0/149.0 kB 16.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.1/45.1 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 74.3 MB/s eta 0:00:00
  Created wheel for FlagEmbedding: filename=FlagEmbedding-1.3.5-py3-none-any.whl size=233746 sha256=d325499a90d333bfc74a48afb0c092aed5db9524e4e205f103ec1675c67fb612
  Stored in directory: /root/.cache/pip/wheels/b2/1f/f6/78f862bb80cb959cc9960b7c4e2d1f702b1bc0e79d19b5f124
  Created wheel for warc3-wet-clueweb09: filename=warc3_wet_clueweb09-0.2.5-py3-none-any.whl size=18919 sha256=b4cc2a4a07c4f509b

First, our imports. After importing `duckdb`, we open a new connection and activate its vector search extension.

In [4]:
from pathlib import Path
import re

import duckdb
from FlagEmbedding import BGEM3FlagModel
import httpx
import numpy as np
import pymupdf4llm
import torch

con = duckdb.connect()
con.install_extension("vss")
con.load_extension("vss")

sql="""SET GLOBAL hnsw_enable_experimental_persistence = true;"""

con.execute(sql)


In [5]:
# Get the handbook, if we don't already have it.
if not Path('handbook.pdf').exists():
    pdfreq = httpx.get('https://www.501commons.org/resources/tools-and-best-practices/human-resources/sample-employee-handbook-national-council-of-nonprofits', verify=False)
    pdf = pdfreq.content
    with open('handbook.pdf', 'wb') as f:
        f.write(pdf)

Below, we divide the handbook into paragraphs, then parse it into a list of dictionaries. We keep track of the section and subsection each paragraph is in, both for the next step and for the LLM's benefit during generation.

In [6]:
# Tried `pdfplumber`, but `pymupdf4llm` works better here.
# `ignore_graphics` keeps it from interpreting random lines as tables.
handbook = pymupdf4llm.to_markdown('handbook.pdf', table_strategy='lines_strict', ignore_graphics=True)

In [7]:
# Get rid of the table of contents and the (empty) mission statement.
handbook = handbook.split('I. MISSION')[2]

In [8]:
# Divide into paragraphs
handbook = handbook.split('\n\n')

In [9]:
section = ''
subsection = ''
paragraphs = []

for para in handbook:
    para = para.strip()
    if re.match(r'[IVX]+\. [\w]+', para):
        para = re.sub(r'[IVX]+\. ', '', para)
        section = para
        subsection = ''
    elif re.match(r'[A-Z]\. [\w]+', para):
        para = re.sub(r'[A-Z]\. ', '', para)
        subsection = para
    elif re.match(r'\[\d+\s?\]', para): # Page numbers. ignore.
        ...
    elif para.strip() == '':
        ...
    else:
        paragraphs.append({
            'section': section,
            'subsection': subsection,
            'paragraph': para
        })

Now that we have our data broken up into dictionaries, loop over them adding a `context` element to each. `context` includes the current paragraph, plus the previous and next paragraphs, if those are in the same section and subsection.

In [10]:
for i in range(len(paragraphs)):
    paragraphs[i]['paragraph'] = re.sub(r'\s+', ' ', paragraphs[i]['paragraph'])
    paragraphs[i]['context'] = paragraphs[i]['paragraph']
    if i > 0 and\
      paragraphs[i]['section'] == paragraphs[i-1]['section'] and\
      paragraphs[i]['subsection'] == paragraphs[i-1]['subsection']:
        paragraphs[i-1]['paragraph'] = re.sub(r'\s+', ' ', paragraphs[i-1]['paragraph'])
        paragraphs[i]['context'] = paragraphs[i-1]['paragraph'] + '\n\n' + paragraphs[i]['context']

    if i < len(paragraphs)-1 and\
      paragraphs[i]['section'] == paragraphs[i+1]['section'] and\
      paragraphs[i]['subsection'] == paragraphs[i+1]['subsection']:
        paragraphs[i+1]['paragraph'] = re.sub(r'\s+', ' ', paragraphs[i+1]['paragraph'])
        paragraphs[i]['context'] = paragraphs[i]['context'] + '\n\n' + paragraphs[i+1]['paragraph']

Print the first few paragraphs to make sure they look like what we expect.

In [11]:
for i in range(10):
  print(paragraphs[i])

{'section': 'OVERVIEW', 'subsection': '', 'paragraph': 'The {ORGANIZATION NAME} Employee Handbook (the “Handbook”) has been developed to provide general guidelines about {ORGANIZATION NAME} policies and procedures for employees. It is a guide to assist you in becoming familiar with some of the privileges and obligations of your employment, including {ORGANIZATION NAME}ʹs policy of voluntary at‐will employment. None of the policies or guidelines in the Handbook are intended to give rise to contractual rights or obligations, or to be construed as a guarantee of employment for any specific period of time, or any specific type of work. Additionally, with the exception of the voluntary at‐will employment policy, these guidelines are subject to modification, amendment or revocation by {ORGANIZATION NAME} at any time, without advance notice.', 'context': 'The {ORGANIZATION NAME} Employee Handbook (the “Handbook”) has been developed to provide general guidelines about {ORGANIZATION NAME} polic

Load an embedding model. Use the GPU if we have one.

In [12]:
device = "cpu"
# use a GPU if available to speed up the embedding computation
if torch.cuda.is_available(): device = "cuda" # Nvidia GPU
elif torch.backends.mps.is_available(): device = "mps" # Apple silicon GPU

model = BGEM3FlagModel('BAAI/bge-m3', use_fp16=True, device=device)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/444 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

...

onnx/model.onnx:   0%|          | 0.00/725k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

onnx/tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/2.27G [00:00<?, ?B/s]

sparse_linear.pt:   0%|          | 0.00/3.52k [00:00<?, ?B/s]

Create a database table to store our data and embeddings, then add the data to it.

In [13]:
con.execute("CREATE SEQUENCE IF NOT EXISTS seq_handbookid START 1")
con.execute("DROP TABLE IF EXISTS handbook")
qry = """CREATE TABLE handbook
(
  id INTEGER PRIMARY KEY DEFAULT NEXTVAL('seq_handbookid'),
  section TEXT,
  subsection TEXT,
  context TEXT,
  embedding FLOAT[1024]
)

"""

con.execute(qry)

In [14]:
for para in paragraphs:
    embedding = model.encode(para['paragraph'])["dense_vecs"]
    qry = f"""INSERT INTO handbook (section, subsection, context, embedding) VALUES (?, ?, ?, ?)"""
    con.execute(qry, (para['section'], para['subsection'], para['context'], embedding))

You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Make sure the `handbook` table has the content we expect.

In [15]:
con.execute('SELECT * FROM handbook LIMIT 5').fetch_df()

,id,section,subsection,context,embedding
0,1,OVERVIEW,,The {ORGANIZATION NAME} Employee Handbook (the...,"[-0.025878906, -0.01033783, -0.051971436, -0.0..."
1,2,OVERVIEW,,The {ORGANIZATION NAME} Employee Handbook (the...,"[0.0021438599, -0.04321289, -0.03894043, 0.004..."
2,3,OVERVIEW,,The personnel polices of {ORGANIZATION NAME} a...,"[-0.0063285828, -0.012916565, -0.059173584, -0..."
3,4,VOLUNTARY AT‐WILL EMPLOYMENT,,Unless an employee has a written employment ag...,"[-0.011520386, 0.0011253357, -0.01486969, 0.00..."
4,5,EQUAL EMPLOYMENT OPPORTUNITY,,{ORGANIZATION NAME} shall follow the spirit an...,"[-0.047790527, -0.022766113, -0.021026611, 0.0..."


Here we create an `embed()` function and add it to the database engine as a UDF. This will simplify querying the database later.

In [16]:
from duckdb.typing import VARCHAR

def embed(sentence: str) -> np.ndarray:
    return model.encode(sentence)['dense_vecs']

con.create_function("embed", embed, [VARCHAR], 'FLOAT[1024]')


qry = "SELECT embed('How much can I drink at work?') AS query_embedding;"
con.execute(qry).fetch_df()

,query_embedding
0,"[-0.017196655, 0.012336731, -0.041107178, -0.0..."


Here, create a simple search function for testing, to see if the naive search results look sensible.

In [28]:
def search(q: str):
    return con.execute("""
        FROM handbook
        SELECT section, subsection, context, array_inner_product(embedding, embed($q)) AS similarity
        ORDER BY similarity DESC
        LIMIT 3""",
        {"q": q}
    ).fetch_df()

drinking = search('How much can i drink at work?')
drinking

,section,subsection,context,similarity
0,SEPARATION,,"- Theft;\n\n- The possession, use, sale or bei...",0.561296
1,"HOURS OF WORK, ATTENDANCE AND PUNCTUALITY",Hours of Work,The normal work week for {ORGANIZATION NAME} s...,0.546338
2,"HOURS OF WORK, ATTENDANCE AND PUNCTUALITY",Overtime,"Overtime pay, which is applicable only to Non‐...",0.546085


In [29]:
print(drinking['context'][0])

- Theft;

- The possession, use, sale or being under the influence of drugs or other controlled substances or alcoholic beverages during working hours or on the {ORGANIZATION NAME} premises at any time in violation of {ORGANIZATION NAME}’s policies.

- Carrying or possessing firearms or weapons on {ORGANIZATION NAME} property;


In [27]:
search('How do i apply for travel expense reimbursement?')['context'][0]

'Employees are responsible for transportation costs between the office and home during normal work hours. Transportation costs are paid by {ORGANIZATION NAME} for work outside normal work hours if the employee is on official business for {ORGANIZATION NAME}. Employees authorized to use their personal cars for {ORGANIZATION NAME} business are reimbursed at the U.S. Internal Revenue Service approved rate.\n\nForms are provided to request reimbursement for actual expenses and advance payment for travel. Receipts must be provided for all expenditures made in order to claim reimbursement.'

### Limitations

* The drinking example only returns a few items from the middle of an itemized list, showing that sometimes the context may not contain all the context the LLM needs. We could attempt to address that with special logic for lists (the paragraph before that list would help), but a better solution might be to let an LLM chunk the document for us and add necessary context to each item.
* There's probably only one useful context for a lot of likely queries to this dataset; an extra layer of LLM-driven relevance checking might help.
